# R2AI Financial Table Q&A — PIPELINE RUNNER (Kaggle GPU T4)
Notebook trung tâm điều phối toàn bộ quy trình End-to-End của dự án trên **Kaggle Notebook** (GPU Tesla T4 16GB VRAM).

### Hai Lựa Chọn Luồng Dữ Liệu:
* **Lựa chọn A (Khuyên dùng - Tiết kiệm thời gian):** Nạp nhanh kho dữ liệu/chỉ mục từ Kaggle Dataset (hỗ trợ cả file nén `.zip.bin` / `.zip` lẫn folder `data/` đã unzip sẵn) rồi chạy thẳng suy luận (`infer`).
* **Lựa chọn B (Build từ đầu - Full Pipeline):** Chạy tuần tự từng bước: **Fetch (Tải BCTC) $\rightarrow$ Corpus (Trích xuất bảng) $\rightarrow$ Index (Tạo chỉ mục) $\rightarrow$ Infer (Suy luận) $\rightarrow$ Package (Đóng gói) $\rightarrow$ Evaluate (Đánh giá)**.

### Cơ chế Resume Checkpoint trên Kaggle:
* Bạn có thể **tải file `questions_pred.json` trực tiếp từ máy tính** lên phần `input`.
* Pipeline sẽ tự động phát hiện và tải file vào thư mục `/kaggle/working/backup/` và `/kaggle/workingoutputs/predictions` để sử dụng và tiếp tục suy luận từ câu bị dừng mà không cần chạy lại từ đầu.

## Mục 1: Khởi tạo Mã nguồn & Cài đặt Môi trường trên Kaggle

In [ ]:
# 1.1. Nhận diện Môi trường & Clone mã nguồn từ GitHub vào /kaggle/working/R2AI-Stage-2
import os, sys, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = "kaggle_web_client" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
IN_LOCAL = not IN_COLAB and not IN_KAGGLE

print(f"Môi trường hiện tại: {'Kaggle' if IN_KAGGLE else ('Google Colab' if IN_COLAB else 'Local')}")

WORKING_DIR = Path("/kaggle/working/R2AI-Stage-2")

if not WORKING_DIR.exists():
    print("Đang clone repository R2AI-Stage-2 vào /kaggle/working/...")
    !git clone https://github.com/Nostagi/R2AI-Stage-2.git /kaggle/working/R2AI-Stage-2
else:
    print("Thư mục làm việc đã tồn tại:", WORKING_DIR)

%cd /kaggle/working/R2AI-Stage-2
print(f"Thư mục hiện tại: {os.getcwd()}")

In [ ]:
# 1.2. Cài đặt các thư viện phụ thuộc chuẩn từ requirements.txt
print("Đang cài đặt dependencies cho GPU T4...")
!pip uninstall -y -q torchaudio
!pip install -q -r requirements.txt
print("Hoàn tất cài đặt môi trường!")

In [ ]:
# 1.3. Kiểm tra GPU CUDA trên Kaggle
import torch
print(f"CUDA Khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Settings -> Accelerator -> Chọn GPU T4 x1.")

--- 
## Mục 2: Chuẩn bị Dữ liệu & Chỉ mục (CHỌN PHƯƠNG ÁN 2.A HOẶC 2.B)

> * Nếu bạn **ĐÃ CÓ** Kaggle Dataset đính kèm (`.zip.bin`, `.zip`, hoặc folder `data`): Chạy **Mục 2.A**.
> * Nếu bạn **CHƯA CÓ** dữ liệu hoặc muốn **BUILD LẠI TỪ ĐẦU**: Bỏ qua 2.A và chạy **Mục 2.B**.

### ⚡ Phương án 2.A: Nạp nhanh từ Kaggle Dataset (Tự động chống lồng thư mục data/data)

In [ ]:
# 2.A. Tự động nạp và giải nén dữ liệu chuẩn vào /kaggle/working/R2AI-Stage-2/data/
# Hỗ trợ: 
# 1) File nén .zip.bin hoặc .zip (tự động nhận diện cấu trúc zip để tránh lỗi data/data)
# 2) Thư mục data/ đã được Kaggle giải nén sẵn
import zipfile, shutil
from pathlib import Path

repo_root = Path("/kaggle/working/R2AI-Stage-2")
data_dir = repo_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

data_loaded = False

if Path("/kaggle/input").exists():
    # --- TRƯỜNG HỢP 1: Tìm file nén .zip.bin hoặc .zip ---
    zip_candidates = list(Path("/kaggle/input").glob("**/*data_backup*.zip.bin")) + \
                     list(Path("/kaggle/input").glob("**/*.zip.bin")) + \
                     list(Path("/kaggle/input").glob("**/*data_backup*.zip")) + \
                     list(Path("/kaggle/input").glob("**/*.zip"))

    if zip_candidates:
        zip_file = zip_candidates[0]
        print(f"[Trường hợp 1] Tìm thấy file nén: {zip_file}")
        print("Đang giải nén dữ liệu...")
        try:
            with zipfile.ZipFile(zip_file, 'r') as zf:
                names = zf.namelist()
                # Nếu trong file zip đã chứa sẵn tiền tố 'data/' -> giải nén tại repo_root để vào thẳng repo_root/data/
                # Nếu trong zip không có tiền tố 'data/' (chứa processed/ trực tiếp) -> giải nén vào data_dir
                has_data_prefix = any(n.startswith("data/") or n.startswith("data\\") for n in names)
                extract_target = repo_root if has_data_prefix else data_dir
                print(f"-> Giải nén vào mục tiêu: {extract_target}")
                zf.extractall(extract_target)
            data_loaded = True
        except Exception as e:
            print(f"Lỗi giải nén trực tiếp: {e} -> Thử copy đổi tên sang .zip...")
            tmp_zip = Path("/kaggle/working/temp_data.zip")
            shutil.copy2(zip_file, tmp_zip)
            with zipfile.ZipFile(tmp_zip, 'r') as zf:
                has_data_prefix = any(n.startswith("data/") for n in zf.namelist())
                extract_target = repo_root if has_data_prefix else data_dir
                zf.extractall(extract_target)
            tmp_zip.unlink(missing_ok=True)
            data_loaded = True

    # --- TRƯỜNG HỢP 2: Tìm thư mục data đã giải nén sẵn trong Dataset ---
    if not data_loaded:
        for input_dir in Path("/kaggle/input").glob("**/*"):
            if input_dir.is_dir() and ((input_dir / "processed").exists() or (input_dir / "index").exists() or (input_dir / "manifest.jsonl").exists()):
                print(f"[Trường hợp 2] Tìm thấy thư mục data sẵn tại: {input_dir}")
                print("Đang đồng bộ dữ liệu vào thư mục làm việc...")
                for item in input_dir.iterdir():
                    dest = data_dir / item.name
                    if item.is_dir():
                        if dest.exists():
                            shutil.rmtree(dest)
                        shutil.copytree(item, dest)
                    else:
                        shutil.copy2(item, dest)
                data_loaded = True
                break

# --- BƯỚC AN TOÀN: Xử lý chống lồng thư mục data/data (Flattening) ---
nested_data = data_dir / "data"
if nested_data.exists() and nested_data.is_dir():
    print("Phát hiện thư mục lồng data/data/ -> Đang tự động chuyển toàn bộ về data/...")
    for item in nested_data.iterdir():
        dest = data_dir / item.name
        if dest.exists():
            if dest.is_dir():
                shutil.rmtree(dest)
            else:
                dest.unlink()
        shutil.move(str(item), str(dest))
    nested_data.rmdir()
    print("Đã dọn dẹp cấu trúc thư mục chuẩn xác!")

if data_loaded:
    print("\n--- KIỂM TRA TÍNH TOÀN VẸN CỦA DỮ LIỆU ---")
    !python main.py verify
else:
    print("Không tìm thấy file .zip.bin hoặc thư mục data trong /kaggle/input/.")
    print("Vui lòng kiểm tra lại dataset đã Add vào Kaggle hoặc chuyển sang Phương án 2.B để build từ đầu.")

### 🔨 Phương án 2.B: Xây dựng toàn bộ Kho dữ liệu & Chỉ mục từ đầu (Full Pipeline Build)

In [ ]:
# 2.B.0 (Stage 00): Tải dữ liệu BCTC thô từ Hugging Face
!python main.py fetch

In [ ]:
# 2.B.1 (Stage 01): Bóc tách OCR, ghép bảng gãy & chuẩn hóa sang 119k file CSV (data/processed/)
!python main.py corpus

In [ ]:
# 2.B.2 (Stage 02): Xây dựng chỉ mục tìm kiếm (BM25 + FAISS Dense BGE-M3 + MetadataStore)
!python main.py index

In [ ]:
# 2.B.3. Kiểm tra tính toàn vẹn của Corpus và Index sau khi build
!python main.py verify

In [ ]:
# 2.B.4 (Tùy chọn): Đóng gói thư mục data/ thành data_backup.zip để lưu trữ lại Kaggle Output
print("Đang nén toàn bộ thư mục data/ thành /kaggle/working/data_backup.zip...")
!zip -q -r /kaggle/working/data_backup.zip data/
print("Đã đóng gói xong data_backup.zip tại /kaggle/working/ sẵn sàng tải về hoặc tạo Dataset!")

--- 
## 🤖 Mục 3: Chạy Suy Luận End-to-End (Stage 03 Infer & Checkpoint Resume)

In [ ]:
# 3.1. Khôi phục Checkpoint từ /kaggle/working/backup/ hoặc Kaggle Dataset nếu có
# Bạn có thể upload file questions_pred.json trực tiếp vào /kaggle/working/backup/
!mkdir -p outputs/predictions /kaggle/working/backup

local_pred = Path("outputs/predictions/questions_pred.json")
backup_pred = Path("/kaggle/working/backup/questions_pred.json")

# 1. Kiểm tra file upload trong /kaggle/working/backup/
if backup_pred.exists() and not local_pred.exists():
    print(f"Tìm thấy checkpoint tải lên tại {backup_pred} -> Đang khôi phục để resume...")
    shutil.copy2(backup_pred, local_pred)

# 2. Kiểm tra file đính kèm trong Kaggle Dataset (/kaggle/input/)
if not local_pred.exists() and Path("/kaggle/input").exists():
    input_preds = list(Path("/kaggle/input").glob("**/*questions_pred*.json"))
    if input_preds:
        print(f"Tìm thấy checkpoint từ Kaggle Dataset: {input_preds[0]} -> Đang nạp để resume...")
        shutil.copy2(input_preds[0], local_pred)

if local_pred.exists():
    import json
    try:
        with open(local_pred, "r", encoding="utf-8") as f:
            data = json.load(f)
            print(f"Sẵn sàng RESUME: Đã có {len(data)}/1012 câu trong checkpoint!")
    except Exception as e:
        print(f"Lỗi đọc checkpoint: {e}")
else:
    print("Chưa có checkpoint cũ -> Sẽ chạy suy luận từ câu đầu tiên (Q1).")

# 3.2. Chạy suy luận toàn bộ 1,012 câu hỏi với mô hình Qwen2.5-Coder-7B-Instruct (4-bit)
# Tự động lưu checkpoint mỗi 5 câu và đồng bộ sang /kaggle/working/backup/questions_pred.json
!python -u main.py infer \
    --questions data/questions/questions.jsonl \
    --model Qwen/Qwen2.5-Coder-7B-Instruct \
    --backend transformers \
    --pred outputs/predictions/questions_pred.json

--- 
## Mục 4: Đóng Gói File Nộp Bài (Stage 04 Package)

In [ ]:
# 4. Đóng gói file nộp bài submission.zip theo đúng định dạng BTC
!python main.py package \
    --pred outputs/predictions/questions_pred.json \
    --out submission.zip

# Sao lưu bản submission.zip và questions_pred.json ra /kaggle/working/ để tải về từ giao diện Kaggle
import shutil
from pathlib import Path
zip_src = Path("outputs/submissions/submission.zip")
if not zip_src.exists():
    zip_src = Path("submission.zip") if Path("submission.zip").exists() else list(Path("outputs/submissions").glob("*.zip"))[0]
shutil.copy2(zip_src, "/kaggle/working/submission.zip")
if Path("outputs/predictions/questions_pred.json").exists():
    shutil.copy2("outputs/predictions/questions_pred.json", "/kaggle/working/questions_pred.json")
print(f"Đã lưu {zip_src} và questions_pred.json tại /kaggle/working/ sẵn sàng tải về!")

--- 
## Mục 5: Đánh Giá Điểm Số (Stage 05 Evaluate)

In [ ]:
# 5. Đánh giá điểm số trên tập nhãn chuẩn gold.json nếu có
if Path("labels/gold.json").exists():
    !python main.py evaluate \
        --gold labels/gold.json \
        --pred outputs/predictions/questions_pred.json
else:
    print("Không tìm thấy labels/gold.json.")